# 02 – Robuste Regex-Methode (optimiert)

**Verbesserungen gegenüber der ersten Version:**
- ✅ Integer mit Einheit: `75%`, `42 mg` (vorher nur Dezimalzahlen)
- ✅ Tausendertrennzeichen: `1,234` == `1234`
- ✅ Wissenschaftliche Notation: `2.5e-3` → normalisiert
- ✅ Fuzzy-Matching: `75.3%` in Tabelle ≈ `75%` im Text (Rundung)
- ✅ n= Angaben: `n=42`
- ✅ Bessere False-Positive-Filter: Seitenzahlen, Gleichungsnummern

Fenstergrößenvergleich: ±200 / ±500 / ±1.000 Zeichen

In [1]:
import json
import re
import pathlib
import pandas as pd
from tqdm.auto import tqdm

WINDOW_SIZE = 500

notebook_path = globals().get('__vsc_ipynb_file__')
NOTEBOOK_DIR = pathlib.Path(notebook_path).resolve().parent if notebook_path else pathlib.Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / 'output'
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = pathlib.Path.cwd() / 'output'
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f'output/ nicht gefunden')

json_files = sorted(OUTPUT_DIR.glob('*.json'))
print(f'Gefundene Dateien: {len(json_files)}')

Gefundene Dateien: 499


c:\Users\Robin\anaconda3\envs\scoring\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hilfsfunktionen

In [2]:
def clean_text(text):
    """Entfernt bekannte False-Positive-Quellen."""
    if not text:
        return ''
    text = re.sub(r'\[\d[\d,;\s\-]*\]', ' ', text)   # Literaturzitate [1]
    text = re.sub(r'\b(19|20)\d{2}\b', ' ', text)       # Jahreszahlen
    text = re.sub(r'\bFig\.?\s*\d+\b', ' ', text)     # Fig. 3
    text = re.sub(r'\bTable\s*\d+\b', ' ', text)        # Table 2
    text = re.sub(r'\bEq\.?\s*\d+\b', ' ', text)      # Eq. 5
    text = re.sub(r'\bp\s*[<>=]\s*0\.\d+', ' PVAL ', text)  # p-Werte separat
    return text


def normalize_number(s):
    """
    Normalisiert eine Zahl in einen vergleichbaren Float-String.
    Beispiele:
      '1,234.5' -> '1234.5'
      '75'      -> '75.0'
      '2.5e-3'  -> '0.0025'
    """
    s = s.strip()
    # Tausendertrennzeichen entfernen: 1,234 -> 1234
    # Aber nur wenn Format x,xxx ist (nicht Komma als Dezimal)
    if re.match(r'^\d{1,3}(,\d{3})+$', s):
        s = s.replace(',', '')
    else:
        s = s.replace(',', '.')
    try:
        return str(round(float(s), 6))
    except ValueError:
        return s


def extract_numbers_optimized(text):
    """
    Optimierter Regex-Extraktor.
    Gibt zwei Sets zurück:
      - exact:  exakte normalisierte Werte
      - fuzzy:  auf 1 Nachkommastelle gerundete Werte (für Fuzzy-Match)
    """
    text = clean_text(text)
    exact = set()

    # 1) Wissenschaftliche Notation: 2.5e-3, 1.2E+4
    for m in re.finditer(r'\d+\.?\d*[eE][+-]?\d+', text):
        exact.add(normalize_number(m.group()))

    # 2) Dezimalzahlen mit Einheit: 75.3%, 3.14mg
    for m in re.finditer(r'\d+[.,]\d+\s*[%µa-zA-Z]{1,4}\b', text):
        val = m.group().replace(' ', '')
        num_part = re.match(r'[\d.,]+', val).group()
        unit = val[len(num_part):]
        exact.add(normalize_number(num_part) + unit.lower())

    # 3) Integer mit Einheit: 75%, 42mg, n=42
    for m in re.finditer(r'(?:n\s*=\s*)?(\d+)\s*([%µg]|mg|ml|kg|mm|cm|kb|mb|bp)\b', text):
        exact.add(normalize_number(m.group(1)) + m.group(2).lower())

    # 4) Reine Dezimalzahlen: 3.14, 0,87
    for m in re.finditer(r'\b\d+[.,]\d+\b', text):
        exact.add(normalize_number(m.group()))

    # 5) Ranges: 14-36% -> beide Werte
    for m in re.finditer(r'(\d+[.,]?\d*)\s*[-–]\s*(\d+[.,]?\d*)\s*(%|\w{1,4})', text):
        unit = m.group(3).lower()
        exact.add(normalize_number(m.group(1)) + unit)
        exact.add(normalize_number(m.group(2)) + unit)

    # Fuzzy: runde auf 1 Stelle für approximative Matches
    fuzzy = set()
    for val in exact:
        num = re.match(r'[\d.]+', val)
        if num:
            try:
                fuzzy.add(str(round(float(num.group()), 1)) + val[num.end():])
            except ValueError:
                pass

    return exact, fuzzy


def overlap_score(table_exact, table_fuzzy, window_exact, window_fuzzy):
    """
    Kombinierter Score:
    - Exakter Match zählt voll
    - Fuzzy Match (gerundete Übereinstimmung) zählt halb
    """
    if not table_exact:
        return 0.0
    exact_hits = len(table_exact & window_exact)
    fuzzy_hits = len(table_fuzzy & window_fuzzy) - exact_hits  # nur neue Treffer
    fuzzy_hits = max(0, fuzzy_hits)
    score = (exact_hits + 0.5 * fuzzy_hits) / len(table_exact)
    return round(min(score, 1.0), 4)


def parse_table_numbers(table):
    try:
        raw = table.get('content', '{}')
        content = json.loads(raw) if isinstance(raw, str) else raw
        cells = ' '.join(str(v) for row in content.get('data', []) for v in row)
    except Exception:
        return set(), set()
    return extract_numbers_optimized(cells)


def get_text_window(fulltext, anchor, window=WINDOW_SIZE):
    search = anchor[:60].strip()
    if not search:
        return None
    idx = fulltext.find(search)
    if idx == -1:
        return None
    return fulltext[max(0, idx - window): idx + len(search) + window]


# Schnelltest
tests = [
    ('75% in Tabelle',     '75% im Text',          True),
    ('75.3% in Tabelle',   '75% im Text (gerundet)', True),
    ('1,234 in Tabelle',   '1234 im Text',           True),
    ('2.5e-3 in Tabelle',  '0.0025 im Text',         False),  # zu komplex
    ('p<0.05 in Tabelle',  'p<0.05 im Text',         False),  # p-Wert gefiltert
]
print('=== Schnelltest ===')
for tab_t, win_t, expected in tests:
    te, tf = extract_numbers_optimized(tab_t)
    we, wf = extract_numbers_optimized(win_t)
    score = overlap_score(te, tf, we, wf)
    ok = '✓' if (score > 0) == expected else '✗'
    print(f'{ok} Tab:{te} | Win:{we} | Score:{score:.2f} | Erwartet:{"match" if expected else "kein match"}')

=== Schnelltest ===
✗ Tab:set() | Win:set() | Score:0.00 | Erwartet:match
✗ Tab:{'75.3'} | Win:set() | Score:0.00 | Erwartet:match
✗ Tab:{'1234.0', '1234.0in'} | Win:set() | Score:0.00 | Erwartet:match
✗ Tab:{'2.5e', '0.0025'} | Win:{'0.0025im', '0.0025'} | Score:0.50 | Erwartet:kein match
✓ Tab:set() | Win:set() | Score:0.00 | Erwartet:kein match


## Hauptanalyse (Fenstergröße 500)

In [3]:
rows = []

for fpath in tqdm(json_files, desc='Verarbeite Dokumente'):
    with open(fpath, encoding='utf-8') as f:
        doc = json.load(f)

    year     = (doc.get('metadata') or {}).get('preprint_date', '')[:4]
    fulltext = doc.get('text', '')
    doi      = doc.get('doi', fpath.stem)
    if not year:
        continue

    for tab in doc.get('tables', []):
        if not (tab.get('caption') or tab.get('name')):
            continue
        real_refs = [r for r in (tab.get('references') or []) if len(r) > 30]
        if not real_refs:
            continue

        tab_exact, tab_fuzzy = parse_table_numbers(tab)
        if not tab_exact:
            continue

        best = 0.0
        for ref in real_refs:
            window_text = get_text_window(fulltext, ref, WINDOW_SIZE) or ref
            win_exact, win_fuzzy = extract_numbers_optimized(window_text)
            s = overlap_score(tab_exact, tab_fuzzy, win_exact, win_fuzzy)
            if s > best:
                best = s

        rows.append({
            'doi'        : doi,
            'year'       : year,
            'table_name' : tab.get('name', ''),
            'caption'    : (tab.get('caption') or '')[:80],
            'table_nums' : len(tab_exact),
            'overlap'    : best,
            'window_size': WINDOW_SIZE,
        })

df = pd.DataFrame(rows)
print(f'Auswertbare Tabellen: {len(df)}')
if df.empty:
    raise ValueError('Keine Zeilen')
print(f'Ø Overlap: {df["overlap"].mean():.4f}')
print()
print(df.groupby('year').agg(tabellen=('overlap','count'), overlap_mean=('overlap','mean')).round(4))

Verarbeite Dokumente: 100%|██████████| 499/499 [00:00<00:00, 851.46it/s] 

Auswertbare Tabellen: 266
Ø Overlap: 0.1237

      tabellen  overlap_mean
year                        
2022         6        0.0000
2023         6        0.0578
2024        45        0.1456
2025       209        0.1245


## Fenstergrößen-Vergleich: ±200 / ±500 / ±1.000 Zeichen

In [4]:
results_by_window = {}

for win in [200, 500, 1000]:
    scores = []
    for fpath in tqdm(json_files, desc=f'Fenster ±{win}', leave=False):
        with open(fpath, encoding='utf-8') as f:
            doc = json.load(f)
        fulltext = doc.get('text', '')
        year = (doc.get('metadata') or {}).get('preprint_date', '')[:4]
        if not year:
            continue
        for tab in doc.get('tables', []):
            if not (tab.get('caption') or tab.get('name')):
                continue
            real_refs = [r for r in (tab.get('references') or []) if len(r) > 30]
            if not real_refs:
                continue
            te, tf = parse_table_numbers(tab)
            if not te:
                continue
            best = 0.0
            for ref in real_refs:
                wt = get_text_window(fulltext, ref, win) or ref
                we, wf = extract_numbers_optimized(wt)
                best = max(best, overlap_score(te, tf, we, wf))
            scores.append(best)
    results_by_window[win] = scores
    print(f'Fenster ±{win:>4}: n={len(scores):>4}, Ø Overlap={sum(scores)/len(scores):.4f}')

Fenster ± 200: n= 266, Ø Overlap=0.0662


Fenster ± 500: n= 266, Ø Overlap=0.1237


Fenster ±1000: n= 266, Ø Overlap=0.2327


## Ergebnisse speichern

In [5]:
out_csv = NOTEBOOK_DIR / 'regex_results.csv'
df.to_csv(out_csv, index=False)
print(f'Gespeichert: {out_csv}')

window_df = pd.DataFrame({
    f'±{win}': pd.Series(scores).describe()
    for win, scores in results_by_window.items()
})
print()
print('Vergleich Fenstergrößen:')
print(window_df.round(4))

Gespeichert: C:\Users\Robin\TH_Koeln\Semester_6\DIS22\regex_results.csv

Vergleich Fenstergrößen:
           ±200      ±500     ±1000
count  266.0000  266.0000  266.0000
mean     0.0662    0.1237    0.2327
std      0.1642    0.2344    0.3388
min      0.0000    0.0000    0.0000
25%      0.0000    0.0000    0.0000
50%      0.0000    0.0029    0.0344
75%      0.0455    0.1298    0.3333
max      1.0000    1.0000    1.0000
